# Diagnóstico del error de SQLite
Este notebook compara el modelo SQLAlchemy con la tabla real `alumno` y corrige la columna faltante `codigo` que provoca el `OperationalError`.

In [ ]:
from sqlalchemy import create_engine, inspect, text
from sqlalchemy.orm import sessionmaker
from app import app
from models import Alumno

engine = create_engine('sqlite:///database.db')
Session = sessionmaker(bind=engine)
session = Session()
print('DB_URL:', engine.url)
print('Model columns:', list(Alumno.__table__.columns.keys()))


In [ ]:
insp = inspect(engine)
print('Actual table columns:', insp.get_columns('alumno'))
print('Names:', [c['name'] for c in insp.get_columns('alumno')])

missing = [c for c in Alumno.__table__.columns.keys() if c not in [col['name'] for col in insp.get_columns('alumno')]]
print('Missing from SQLite:', missing)


In [ ]:
from sqlalchemy import text

expected = ['codigo', 'a_paterno', 'a_materno', 'sexo', 'comuna', 'nacionalidad', 'f_nacimiento', 'asistencia']
actual = [c['name'] for c in inspect(engine).get_columns('alumno')]
print('Root cause: migration was skipped or DB was created before model update.')
for col in expected:
    if col not in actual:
        session.execute(text(f'ALTER TABLE alumno ADD COLUMN {col} VARCHAR(100)'))
        print('Added:', col)
        actual.append(col)

session.commit()
print('Updated schema:', [c['name'] for c in inspect(engine).get_columns('alumno')])


In [ ]:
result = session.execute(text('SELECT alumno.id, alumno.codigo, alumno.nombre, alumno.apellido FROM alumno LIMIT 5')).fetchall()
print('Query result sample:', result)
